# Applicability Domain

This notebook accompanies the QSAR tutorial chapter: **Applicability Domain**.

## Prepare data and train a model

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors

def find_example_dataset():
    candidates = [
        Path("../data/example_qsar_dataset.csv"),
        Path("data/example_qsar_dataset.csv"),
        Path("../../data/example_qsar_dataset.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find example_qsar_dataset.csv.")

def mol_from_smiles(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def calculate_all_rdkit_descriptors(mol):
    return Descriptors.CalcMolDescriptors(mol, missingVal=np.nan)

# Load data
df = pd.read_csv(find_example_dataset())

# Convert SMILES to RDKit molecules and remove invalid molecules
df["Mol"] = df["SMILES"].apply(mol_from_smiles)
df_clean = df.dropna(subset=["Mol"]).copy()

# Calculate all default RDKit descriptors
descriptor_rows = [
    calculate_all_rdkit_descriptors(mol)
    for mol in df_clean["Mol"]
]

X = pd.DataFrame(descriptor_rows, index=df_clean.index)

# Convert to numeric and remove descriptor columns containing missing values
X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1)

# Target values
y = df_clean["Activity"].astype(float)

print("Number of valid molecules:", len(df_clean))
print("Number of RDKit descriptors:", X.shape[1])

X.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

selector = VarianceThreshold(threshold=1e-8)
X_var = pd.DataFrame(
    selector.fit_transform(X),
    columns=X.columns[selector.get_support()],
    index=X.index
)

top_features = X_var.var().sort_values(ascending=False).head(12).index
X_model = X_var[top_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y,
    test_size=0.25,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

model.fit(X_train_scaled, y_train)

y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

print("Training R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))

## Williams plot calculations

In [ ]:
# Number of training compounds and descriptors
n_train = X_train_scaled.shape[0]
p = X_train_scaled.shape[1]

# Add intercept column for leverage calculation
X_train_aug = np.column_stack([
    np.ones(n_train),
    X_train_scaled
])

X_test_aug = np.column_stack([
    np.ones(X_test_scaled.shape[0]),
    X_test_scaled
])

# Training leverage
hat_matrix_train = (
    X_train_aug
    @ np.linalg.pinv(X_train_aug.T @ X_train_aug)
    @ X_train_aug.T
)

leverage_train = np.diag(hat_matrix_train)

# Test leverage
hat_matrix_test = (
    X_test_aug
    @ np.linalg.pinv(X_train_aug.T @ X_train_aug)
    @ X_test_aug.T
)

leverage_test = np.diag(hat_matrix_test)

# Residuals
residuals_train = y_train - y_train_pred
residuals_test = y_test - y_test_pred

# Residual standard deviation from training set
s = np.sqrt(
    np.sum(residuals_train ** 2) / (n_train - p - 1)
)

std_res_train = residuals_train / s
std_res_test = residuals_test / s

# Warning leverage threshold
h_star = 3 * (p + 1) / n_train

print("n_train:", n_train)
print("p:", p)
print("Warning leverage h*:", h_star)

## Draw Williams plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

plt.scatter(
    leverage_train,
    std_res_train,
    label="Training compounds",
    alpha=0.7
)

plt.scatter(
    leverage_test,
    std_res_test,
    label="Test compounds",
    alpha=0.7
)

plt.axhline(y=3, linestyle="--")
plt.axhline(y=-3, linestyle="--")
plt.axvline(x=h_star, linestyle="--")

plt.xlabel("Leverage")
plt.ylabel("Standardized residual")
plt.title("Williams Plot")
plt.legend()
plt.show()

## Flag compounds outside the applicability domain

In [ ]:
train_ad = pd.DataFrame({
    "Set": "Training",
    "Name": df_clean.loc[y_train.index, "Name"].values,
    "Leverage": leverage_train,
    "StandardizedResidual": np.asarray(std_res_train),
})

test_ad = pd.DataFrame({
    "Set": "Test",
    "Name": df_clean.loc[y_test.index, "Name"].values,
    "Leverage": leverage_test,
    "StandardizedResidual": np.asarray(std_res_test),
})

ad_table = pd.concat([train_ad, test_ad], ignore_index=True)

ad_table["HighLeverage"] = ad_table["Leverage"] > h_star
ad_table["LargeResidual"] = ad_table["StandardizedResidual"].abs() > 3
ad_table["OutsideAD"] = ad_table["HighLeverage"] | ad_table["LargeResidual"]

display(ad_table.sort_values("OutsideAD", ascending=False).head(15))

## Similarity-based AD example with Tanimoto similarity

In [ ]:
from rdkit.Chem import AllChem, DataStructs

# Create Morgan fingerprints for training and test compounds
train_mols = df_clean.loc[y_train.index, "Mol"].tolist()
test_mols = df_clean.loc[y_test.index, "Mol"].tolist()

train_fps = [
    AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    for mol in train_mols
]

test_fps = [
    AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    for mol in test_mols
]

max_similarities = []

for fp in test_fps:
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    max_similarities.append(max(sims))

similarity_ad = pd.DataFrame({
    "Name": df_clean.loc[y_test.index, "Name"].values,
    "MaxTanimotoToTraining": max_similarities
}).sort_values("MaxTanimotoToTraining")

display(similarity_ad)